# Benchmark row: RF-DETR-Medium (DINOv2 backbone) on the v2 folds and the official split

Foundation-backbone detector for the benchmark (docs/BENCHMARK_V2_PLAN.md, fixed before any RF-DETR result). Same lists, seeds and fixed-epoch protocol as the YOLO / RT-DETR rows:
rfdetr 1.11.2, RF-DETR-Medium (Apache 2.0), COCO/Objects365-pretrained checkpoint, input 640, 100 epochs, no early stopping, EMA on, last-epoch EMA weights, batch 4 x grad-accum 4 (= 16), lr 1e-4 / encoder 1.5e-4, package defaults otherwise.
Scoring: `scripts/evaluation/offline_eval.py`, an exact replica of the Ultralytics 8.4.215 metric (verified: `scripts/evaluation/test_offline_eval_parity.py`), so numbers are on the same scale as YOLO / RT-DETR.

**Step 1: `SMOKE = True`** (about 15-20 min): installs rfdetr, downloads the checkpoint, trains 2 epochs on fold 0. Paste the printed table to Claude (time per epoch and GPU memory decide the parts).
**Step 2: `SMOKE = False`, `PART = "A"`, `"B"`, `"C"`** as separate Save Versions (18 runs: 15 v2 + 3 official). A time guard skips any run that would not finish inside the 12 h session and prints it for a later part.

In [ ]:
import os, sys, subprocess
REPO_URL = "https://github.com/nithoukhan1/hbfmid.git"
REPO = "/kaggle/working/hbfmid"
if os.path.exists(REPO):
    subprocess.run(["git", "-C", REPO, "pull", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO], check=True)
import torch
TORCH_BEFORE = torch.__version__
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "rfdetr[train]==1.11.2"], check=True)
os.environ["WANDB_MODE"] = "disabled"
from importlib.metadata import version
print("rfdetr", version("rfdetr"), "| torch before/after install:", TORCH_BEFORE, version("torch"))
print("GPUs:", [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])
assert torch.cuda.is_available(), "No GPU: set Accelerator to GPU T4 x2, then restart."
print("repo commit:", subprocess.run(["git", "-C", REPO, "log", "--oneline", "-1"], capture_output=True, text=True).stdout.strip())
assert os.path.exists(f"{REPO}/scripts/rfdetr_bench/train_rfdetr.py"), "repo is behind: push scripts/rfdetr_bench first"
# download the pretrained checkpoint ONCE (cwd is inherited by the workers) so the two GPU workers do not race
# (in a throwaway subprocess, so this notebook kernel does not keep a model in GPU memory during the runs)
subprocess.run([sys.executable, "-c", "from rfdetr import RFDETRMedium; RFDETRMedium(resolution=640)"], check=True)
print("checkpoints in cwd:", [f for f in os.listdir(".") if f.endswith((".pth", ".pt"))])

In [ ]:
DATA_ROOT = "/kaggle/input/datasets/nithoukhan2/hbfmid/Human Bone Fractures Multi-modal Image Dataset (HBFMID)/Bone Fractures Detection"
assert os.path.isdir(DATA_ROOT), DATA_ROOT
SMOKE = True                   # True first; then False with PART = "A", "B", "C"
PART = "A"
V2 = f"{REPO}/data_splits_v2"; V1 = f"{REPO}/data_splits"          # official lists live in data_splits/official
EST_MIN = 160                  # first guess of minutes per v2 run (2 runs in parallel); replaced by the measured times as runs finish
LIMIT_MIN = 690                # stop launching runs that would end after 11.5 h of session time
PARTS = {"A": [("official", 0, s) for s in (42, 43, 44)] + [("clean", 0, 42)],
         "B": [("clean", f, 42) for f in (1, 2, 3, 4)] + [("clean", f, 43) for f in (0, 1, 2)],
         "C": [("clean", f, 43) for f in (3, 4)] + [("clean", f, 44) for f in (0, 1, 2, 3, 4)],
         "D": []}             # leftovers printed by an earlier part go here, e.g. [("clean", 4, 44)]
if SMOKE:
    EPOCHS, OUT, ZIP, RUNS = 2, "/kaggle/working/results/rfd_smoke", "rfd_smoke_results", [("clean", 0, 42)]
else:
    EPOCHS, OUT, ZIP, RUNS = 100, f"/kaggle/working/results/rfd_{PART}", f"rfd_{PART}_results", PARTS[PART]

def make_cmd(mode, fold, seed):
    return [sys.executable, f"{REPO}/scripts/rfdetr_bench/train_rfdetr.py", "--mode", mode, "--fold", str(fold), "--seed", str(seed),
            "--size", "medium", "--resolution", "640", "--epochs", str(EPOCHS), "--batch", "4", "--accum", "4",
            "--data-root", DATA_ROOT, "--splits-dir", V1 if mode == "official" else V2, "--out", OUT]
TASKS = [(f"{m}_f{f}_s{s}_rfdetr-medium" if m == "clean" else f"official_s{s}_rfdetr-medium", m, make_cmd(m, f, s)) for m, f, s in RUNS]
print(len(TASKS), "runs, epochs =", EPOCHS, "->", OUT); [print(" ", t[0]) for t in TASKS]

In [ ]:
import queue, threading, time
os.makedirs(f"{OUT}/logs", exist_ok=True)
gpus = list(range(torch.cuda.device_count()))
q = queue.Queue()
for t in TASKS:
    q.put(t)
failed, skipped, done_min = [], [], []
T_START = time.time()

def worker(gpu):
    while True:
        try:
            name, mode, cmd = q.get_nowait()
        except queue.Empty:
            return
        est = max([EST_MIN] + done_min) * (1.5 if mode == "official" else 1.0)
        if (time.time() - T_START) / 60 + est > LIMIT_MIN:
            skipped.append(name); print(f"[GPU {gpu}] {name}: SKIPPED (would end after {LIMIT_MIN} min)", flush=True); continue
        t0 = time.time()
        env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(gpu))
        with open(f"{OUT}/logs/{name}.log", "w") as f:
            rc = subprocess.run(cmd, stdout=f, stderr=subprocess.STDOUT, env=env).returncode
        mins = (time.time() - t0) / 60
        if rc == 0 and mode == "clean":
            done_min.append(mins)
        print(f"[GPU {gpu}] {name}: {'OK' if rc == 0 else 'FAILED rc=' + str(rc)} in {mins:.1f} min", flush=True)
        if rc != 0:
            failed.append(name)

threads = [threading.Thread(target=worker, args=(g,)) for g in gpus]
[t.start() for t in threads]; [t.join() for t in threads]
print("failed runs:", failed or "none"); print("skipped runs (put them in PARTS['D']):", skipped or "none")
for n in failed:
    print("\n==== tail of", n)
    print("".join(open(f"{OUT}/logs/{n}.log").readlines()[-30:]))

In [ ]:
import glob, json, shutil
print(f"{'run':40s} {'mAP50-95':>9s} {'mAP50':>7s} {'macro':>7s} {'train_min':>9s} {'min/epoch':>9s} {'GPU GB':>6s} {'cls off':>7s}  flag")
for p in sorted(glob.glob(f"{OUT}/*/metrics.json")):
    r = json.load(open(p)); m = r["test"]["map5095"]; mac = r["test"]["macro_eligible_ap5095"] or 0
    flag = "<-- collapsed? check the log" if m < 0.01 else ""
    print(f"{r['tag']:40s} {m:9.4f} {r['test']['map50']:7.4f} {mac:7.4f} {r['train_seconds']/60:9.1f} {r['train_seconds']/60/r['epochs_cfg']:9.2f} "
          f"{str(r.get('gpu_mem_peak_gb')):>6s} {r.get('class_id_offset')!s:>7s}  {flag}")
if SMOKE:
    print("\nSMOKE: min/epoch x 100 = minutes per real run (2 runs share the machine). Paste this table to Claude.")
shutil.make_archive(f"/kaggle/working/{ZIP}", "zip", OUT)
print("\nzip:", f"/kaggle/working/{ZIP}.zip"); print(sorted(os.listdir(OUT)))
if failed:
    print("\nWARNING: failed runs:", failed)

After the run: download `rfd_<PART>_results.zip` (or `rfd_smoke_results.zip`), unzip once into `results/` (any folder name; Claude renames it), and paste the printed table to Claude.